# Advanced Model Comparison

## Objective

Determine whether a gradient boosting model can outperform the current best model.

## Current Best Model

Tuned Random Forest

- MAE = 17.2889 mg/dL
- RMSE = 25.6017 mg/dL
- R² = 0.8266

## Candidate Model

HistGradientBoostingRegressor

The same patient-wise chronological test split will be used for a fair comparison.

In [1]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
DATA_PATH = "../data/processed/ml_dataset.csv"

df = pd.read_csv(DATA_PATH)

df["timestamp"] = pd.to_datetime(df["timestamp"])

df = df.sort_values(
    ["Patient", "timestamp"]
).reset_index(drop=True)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

Dataset loaded successfully.
Shape: (65499, 21)


In [3]:
feature_columns = [
    "glucose_value",
    "glucose_lag_1",
    "glucose_lag_2",
    "glucose_lag_3",
    "glucose_lag_6",
    "glucose_change_5min",
    "glucose_change_15min",
    "meal_carbs",
    "bolus_dose",
    "basal_value",
    "temp_basal_value",
    "exercise_duration",
    "exercise_intensity",
    "sleep_duration_hours",
    "sleep_quality",
    "finger_stick_value",
    "hour",
    "day_of_week"
]

target_column = "target_glucose_30min"

print("Features:", len(feature_columns))
print("Target:", target_column)

Features: 18
Target: target_glucose_30min


In [4]:
train_parts = []
test_parts = []

for patient in df["Patient"].unique():

    patient_df = df[
        df["Patient"] == patient
    ].sort_values("timestamp")

    split_index = int(len(patient_df) * 0.80)

    train_parts.append(
        patient_df.iloc[:split_index]
    )

    test_parts.append(
        patient_df.iloc[split_index:]
    )

train_df = pd.concat(
    train_parts
).reset_index(drop=True)

test_df = pd.concat(
    test_parts
).reset_index(drop=True)

print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)

Training shape: (52396, 21)
Testing shape: (13103, 21)


In [5]:
X_train = train_df[feature_columns]
y_train = train_df[target_column]

X_test = test_df[feature_columns]
y_test = test_df[target_column]

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (52396, 18)
X_test : (13103, 18)
y_train: (52396,)
y_test : (13103,)


In [6]:
print("Training HistGradientBoosting...")

hgb_model = HistGradientBoostingRegressor(
    max_iter=300,
    learning_rate=0.05,
    max_leaf_nodes=31,
    min_samples_leaf=20,
    l2_regularization=1.0,
    random_state=42
)

hgb_model.fit(
    X_train,
    y_train
)

print("HistGradientBoosting training completed.")

Training HistGradientBoosting...
HistGradientBoosting training completed.


In [7]:
hgb_predictions = hgb_model.predict(
    X_test
)

print("Predictions generated successfully.")
print("Number of predictions:", len(hgb_predictions))

Predictions generated successfully.
Number of predictions: 13103


In [8]:
hgb_mae = mean_absolute_error(
    y_test,
    hgb_predictions
)

hgb_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        hgb_predictions
    )
)

hgb_r2 = r2_score(
    y_test,
    hgb_predictions
)

print("HistGradientBoosting Performance")
print("=" * 40)

print(f"MAE  : {hgb_mae:.4f} mg/dL")
print(f"RMSE : {hgb_rmse:.4f} mg/dL")
print(f"R²   : {hgb_r2:.4f}")

HistGradientBoosting Performance
MAE  : 17.2398 mg/dL
RMSE : 25.7510 mg/dL
R²   : 0.8246


In [9]:
comparison = pd.DataFrame({
    "Model": [
        "Tuned Random Forest",
        "HistGradientBoosting"
    ],
    "MAE": [
        17.2889,
        hgb_mae
    ],
    "RMSE": [
        25.6017,
        hgb_rmse
    ],
    "R²": [
        0.8266,
        hgb_r2
    ]
})

comparison

,Model,MAE,RMSE,R²
0,Tuned Random Forest,17.288900,25.601700,0.826600
1,HistGradientBoosting,17.239825,25.751024,0.824591


In [10]:
if (
    hgb_r2 > 0.8266
    and hgb_mae < 17.2889
):

    print("🏆 HistGradientBoosting is the new best model.")

elif hgb_r2 > 0.8266:

    print(
        "HistGradientBoosting has better R² "
        "but not better MAE."
    )

elif hgb_mae < 17.2889:

    print(
        "HistGradientBoosting has better MAE "
        "but not better R²."
    )

else:

    print(
        "🏆 Tuned Random Forest remains the best model."
    )

HistGradientBoosting has better MAE but not better R².


In [11]:
if (
    hgb_r2 > 0.8266
    and hgb_mae < 17.2889
):

    hgb_path = (
        "../models/"
        "hist_gradient_boosting_glucose_model.pkl"
    )

    joblib.dump(
        hgb_model,
        hgb_path
    )

    print(
        "Improved HistGradientBoosting model saved:"
    )
    print(hgb_path)

else:

    print(
        "HistGradientBoosting did not outperform "
        "the tuned Random Forest."
    )

HistGradientBoosting did not outperform the tuned Random Forest.
